# ScalingAI Apps

**Module 12 · Lesson 12.7**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Why AI Apps Scale Differently
- Horizontal vs Vertical, and Stateless
- Autoscale on Concurrency, Not CPU
- Cold Starts, Min and Max Instances
- Queue-Based Load Leveling
- Scaling the Model Tier: GPU Batching

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## 10x traffic, CPU at 15 percent, zero new instances

> 💡 **Info**
>
> The launch went viral. Traffic jumped 10x in an hour, users started timing out - and when you opened the dashboard, **CPU was sitting at 15 percent** and the autoscaler had added **zero instances**. It was doing exactly what you told it: scale when CPU is busy. But your app was not CPU-busy; it was **I/O-busy**, sitting with hundreds of requests all *waiting* on the model API at once, using almost no CPU. That is the whole problem with scaling AI apps: **they break the assumptions a normal web app makes**. You scale on the wrong signal, you forget the model tier is a different animal, and you discover the hard way that you cannot scale past the provider’s rate limit. This lesson builds the scaling toolkit - concurrency-based autoscaling, min and max instances, queue-based load leveling, GPU batching, and backpressure at the ceiling - and you can run every piece with no cloud, no GPU, and no API key.

### 🎯 What you will be able to do after this lesson

- **Build** a scaling setup for an AI app - concurrency-based autoscaling, min/max instances, a load-leveling queue, and backpressure - as runnable models plus an illustrative Cloud Run + vLLM/KEDA config.

- **Compare** horizontal vs vertical scaling, CPU vs concurrency as the autoscaling signal, and an I/O-bound vs a compute-bound concurrency setting.

- **Operate** cold starts (min-instances), a downstream cap (max-instances), and queue-depth worker autoscaling.

- **Evaluate** a scaling design: right signal, stateless, warm, capped to the provider, buffered by a queue, and graceful at the ceiling?

> 📦 **Info**
>
> ✅ Before you startIn **12.5** you learned a container is ephemeral and stateless - that is exactly what makes horizontal scaling possible (any instance serves any request). In **12.3** the gateway enforced the provider’s RPM/TPM limits - that same limit is the hard ceiling you cannot scale past. Monitoring the scaling signals is **Lesson 12.8**; cost at scale is **Module 13**; the backpressure degradation mechanics are **Lesson 12.2**.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 📞 **Analogy**
>
> Scaling an AI app is running a **call centre where the agents spend most of their time on hold**. In a normal shop, a busy agent is a busy agent - you watch how hard they are working (CPU) and hire more when they are flat out. But your AI agents mostly sit *waiting* on a supplier at the other end of the line (the model API), phone to ear, barely lifting a finger. Watch their effort and you will think the floor is quiet while the queue out the door grows. You have to count the **calls in progress** (concurrency), not the effort. **Where it breaks down:** a call centre can always hire more agents; you cannot dial the supplier more times than your contract allows - that is the rate-limit ceiling.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“Autoscale on CPU, like any web app.”** An I/O-bound LLM app keeps CPU low while it is saturated, so a CPU autoscaler never fires. Scale on **concurrency** (in-flight requests) instead.
> - **“More instances always means more throughput.”** Past the provider’s rate limit, extra instances just collect 429s. You scale the bottleneck, then apply backpressure at the ceiling.

> 💡 **Info**
>
> ⚠️ Anti-patternTwo settings that will bite. **High concurrency on a compute-bound on-instance model** - many requests fight over one GPU, so they all go slow or run out of memory (an on-instance model wants *low* concurrency). And **stateful instances** that keep session or state on local disk - the moment a request lands on a different instance, it is gone. Set concurrency by workload, and keep instances stateless with state in a shared store.

---

## 🎯 Concept 1: Why AI Apps Scale Differently

### Why AI Apps Scale Differently

An LLM app that calls a model is I/O-bound: it spends each request waiting, so CPU stays low even when saturated. A CPU autoscaler never fires; you must scale on concurrency, and the real ceiling is the provider’s rate limit.

Start with why the normal playbook fails. A traditional web app is **CPU-bound** - more traffic means more work per second means higher CPU, so watching CPU is a fine proxy for “we are busy, add instances.” An LLM app that calls a model API is the opposite: it is **I/O-bound**. Each request spends almost all its time *waiting* on the model to respond over the network, using barely any CPU. So an instance can be completely **saturated** - hundreds of requests in flight, all waiting - while CPU sits at 15 or 30 percent. A CPU-based autoscaler looks at that and does nothing, and your users time out. The fix is to scale on the signal that actually reflects load: **concurrency**, the number of in-flight requests per instance. And there is a second twist: the real ceiling is often not your machines at all but the **provider’s rate limit** (from 12.3). The block shows a CPU autoscaler sleeping through a spike, keyless.

> ☎️ **Analogy**
>
> It is the **call centre on hold** from the warm-up, made concrete. Ten agents, each with a customer on the line but every call parked on hold waiting for a supplier - the agents are idle, effort near zero, but every line is busy and new callers get a busy tone. Measure effort and you send everyone home; measure *lines in use* and you realize you are full and need more lines. AI instances are those agents: mostly waiting, barely working, yet completely full.

Your LLM app is at 300 in-flight per instance and users are timing out, but CPU reads 30 percent. What does a CPU-based autoscaler do?

**📝 `01_why_different.py`** - *runnable*

In [ ]:
# An LLM-API instance spends most of each request WAITING on the model (I/O-bound),
# so CPU stays LOW even when the instance is saturated with in-flight requests.
CPU_PER_INFLIGHT = 0.001    # a waiting request barely uses CPU
CPU_TARGET = 0.60           # a CPU autoscaler fires above 60%
CONCURRENCY_TARGET = 80     # a concurrency autoscaler fires above 80 in-flight per instance

def decide(inflight):
    cpu = min(1.0, inflight * CPU_PER_INFLIGHT)
    cpu_scaler = "SCALE UP" if cpu > CPU_TARGET else "asleep (cpu below 60%)"
    conc_scaler = "SCALE UP" if inflight > CONCURRENCY_TARGET else "hold"
    return cpu, cpu_scaler, conc_scaler

print("An I/O-bound LLM app under a traffic spike (in-flight requests per instance):")
for inflight in [40, 120, 300]:
    cpu, cs, cc = decide(inflight)
    print("  in-flight {:>3}:  CPU {:>2.0f}%   CPU-autoscaler: {:<22}  concurrency-autoscaler: {}".format(
        inflight, cpu * 100, cs, cc))
print()
print("At 300 in-flight the instance is drowning, but CPU is only 30%: a CPU autoscaler never scales up.")
print("Scale an I/O-bound LLM app on CONCURRENCY (in-flight requests), not CPU.")

# Output:
# An I/O-bound LLM app under a traffic spike (in-flight requests per instance):
#   in-flight  40:  CPU  4%   CPU-autoscaler: asleep (cpu below 60%)  concurrency-autoscaler: hold
#   in-flight 120:  CPU 12%   CPU-autoscaler: asleep (cpu below 60%)  concurrency-autoscaler: SCALE UP
#   in-flight 300:  CPU 30%   CPU-autoscaler: asleep (cpu below 60%)  concurrency-autoscaler: SCALE UP
#
# At 300 in-flight the instance is drowning, but CPU is only 30%: a CPU autoscaler never scales up.
# Scale an I/O-bound LLM app on CONCURRENCY (in-flight requests), not CPU.

- Each in-flight request barely uses CPU (it is **waiting** on the model), so CPU stays low as load climbs.
- At **300 in-flight** the instance is drowning, but CPU is only **30 percent** - below the 60-percent trigger.
- The **CPU autoscaler stays asleep** at every level; the **concurrency autoscaler** fires once in-flight passes its target.
- The lesson: for an I/O-bound LLM app, scale on **concurrency** (in-flight requests), not CPU - and remember the provider’s rate limit is the real ceiling.

#### 💡 What just happened

⚡ What just happened? An LLM app that calls a model is I/O-bound: it waits most of each request, so CPU stays low even when saturated and a CPU autoscaler never fires. Tradeoff / the framing for the lesson: scale on concurrency (in-flight requests), not CPU, and know that the provider’s rate limit - not your machines - is often the true ceiling. Every later step is a piece of scaling that signal correctly.

- A traffic dial 1x to 10x; a CPU gauge barely moves while an in-flight meter climbs
- The CPU autoscaler stays asleep; the concurrency autoscaler wakes and adds instances

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Horizontal vs Vertical, and Stateless

### Horizontal vs Vertical, and Stateless

Scale UP (a bigger instance) hits a ceiling and is a single point of failure; scale OUT (more instances) is linear and resilient, and it is the default for the web tier - but only if instances are stateless.

There are two directions to add capacity. **Vertical** scaling makes one instance *bigger* - more CPU, more RAM, a bigger GPU. It is simple, but it has a hard ceiling (there is a biggest machine) and that one instance is a **single point of failure**. **Horizontal** scaling adds *more* instances behind a load balancer - capacity grows linearly, and if one dies the others carry on. For the stateless web/API tier, horizontal is the default. But horizontal scaling has one hard requirement: the instances must be **stateless**, so that *any* instance can serve *any* request. If an instance keeps a user’s session on its own local disk, the moment the load balancer sends that user’s next request to a different instance, the session is gone. Keep state in a **shared store** (a database, a cache) and every instance is interchangeable (this is exactly why 12.5’s containers are ephemeral). The block shows the two directions and the stateless requirement, keyless.

> 🚚 **Analogy**
>
> It is a **bigger truck versus more trucks**. To move more cargo you can buy one enormous truck (vertical) - but there is a largest truck money can buy, and if it breaks down everything stops. Or you run a *fleet* of ordinary trucks (horizontal) - add as many as you need, and one breakdown barely dents the fleet. The catch: a fleet only works if any driver can take any load. If the paperwork for a shipment is locked in one specific truck’s glovebox (local state), no other truck can deliver it. Put the paperwork in a shared office (a shared store) and every truck is interchangeable.

You scale your web tier to 10 instances, but users randomly get logged out. Sessions are stored on each instance’s local disk. What is happening?

**📝 `02_horizontal_vertical.py`** - *runnable*

In [ ]:
# Vertical = a bigger instance. Horizontal = more instances. Horizontal needs STATELESS.
def capacity(instances, per_instance_rps):
    return instances * per_instance_rps

print("Scaling a web/API tier - two directions:")
print("  vertical   (1 bigger instance):  {:>4} req/s  -> ceiling = the biggest machine; a single point of failure".format(capacity(1, 500)))
print("  horizontal (10 instances):       {:>4} req/s  -> linear capacity, no single point of failure".format(capacity(10, 100)))
print()

# Horizontal only works if instances are STATELESS: any instance can serve any request.
def serve(req_instance, session_home, store):
    if store == "local":
        return "OK" if req_instance == session_home else "FAIL: session is on a different instance"
    return "OK: session lives in a shared store, so any instance serves it"

print("Horizontal scaling requires STATELESS instances:")
print("  local session,  request routed to a DIFFERENT instance  -> " + serve("B", "A", "local"))
print("  shared session, request routed to a DIFFERENT instance  -> " + serve("B", "A", "shared"))
print()
print("Scale the STATELESS web/API tier OUT (horizontal); scale the model tier UP (bigger GPU) or out (more replicas).")

# Output:
# Scaling a web/API tier - two directions:
#   vertical   (1 bigger instance):   500 req/s  -> ceiling = the biggest machine; a single point of failure
#   horizontal (10 instances):       1000 req/s  -> linear capacity, no single point of failure
#
# Horizontal scaling requires STATELESS instances:
#   local session,  request routed to a DIFFERENT instance  -> FAIL: session is on a different instance
#   shared session, request routed to a DIFFERENT instance  -> OK: session lives in a shared store, so any instance serves it
#
# Scale the STATELESS web/API tier OUT (horizontal); scale the model tier UP (bigger GPU) or out (more replicas).

- **Vertical** (one bigger instance) has more capacity but a ceiling and a single point of failure.
- **Horizontal** (ten instances) scales linearly and survives an instance dying.
- Horizontal only works if instances are **stateless**: a **local** session fails when the request lands on a different instance.
- A **shared-store** session works from any instance - so scale the stateless web tier **out**, and the model tier **up** (bigger GPU) or out (more replicas).

#### 💡 What just happened

⚡ What just happened? Vertical scaling makes an instance bigger (a ceiling, a single point of failure); horizontal scaling adds instances (linear, resilient) and is the default for the web tier - but only if instances are stateless, with state in a shared store. Tradeoff: horizontal scaling needs stateless design and a load balancer, in exchange for near-unlimited, resilient capacity. Next: how the autoscaler decides how many instances to run.

- Grow one instance (vertical, hits a ceiling) vs add instances behind a balancer (horizontal)
- Route a request to a different instance: a local session fails, a shared-store session works

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Autoscale on Concurrency, Not CPU

### Autoscale on Concurrency, Not CPU

Cloud Run scales on request concurrency. Set the concurrency per instance by workload: high for an I/O-bound app that calls an API, low for a compute-bound model that runs on the instance.

Now the setting that everything turns on. **Cloud Run** (and modern autoscalers) scale on request **concurrency** - it aims to keep each instance at roughly 60 percent of the maximum concurrent requests you configure with `--concurrency`. The value you choose depends entirely on the workload. For an **I/O-bound** app that calls an external LLM API, set concurrency **high** (80 to 200): each request is mostly waiting, so one instance can hold many at once, and a high value means you need far fewer instances. For a **compute-bound** app that runs a model *on the instance*, set concurrency **low** (1 to 4): each request saturates the CPU or GPU, so more than a few at a time just makes them all slow (or runs out of memory). Getting this backwards is the classic AI scaling bug. The block sizes both, keyless.

> 🍽️ **Analogy**
>
> It is a restaurant **sizing its tables to the meal, not the guest count**. For quick coffees (I/O-bound - people mostly sit and wait), you seat many per table and turn them fast; a few big tables serve a crowd. For a twelve-course tasting menu (compute-bound - each diner monopolizes the kitchen), you seat one party per table and the kitchen handles only a few at a time; cram more in and every dish comes out late and cold. Same room, opposite seating - and picking the wrong one ruins the night.

Your app just calls the Anthropic API and waits for the response. Should each instance handle many concurrent requests or just one?

**📝 `03_concurrency.py`** - *runnable*

In [ ]:
# Cloud Run scales on request CONCURRENCY (target ~60% of the max you set).
# The right --concurrency depends on whether the app WAITS (I/O) or COMPUTES.
WORKLOADS = {
    "calls an LLM API (I/O-bound)":        {"concurrency": 80, "note": "one instance holds many waiting requests"},
    "runs a model on-instance (compute)":  {"concurrency": 4,  "note": "each request saturates the CPU/GPU"},
}
def instances_needed(concurrent, per_instance):
    return -(-concurrent // per_instance)   # ceiling division

print("Serving 240 concurrent requests - set --concurrency by workload:")
for name, w in WORKLOADS.items():
    n = instances_needed(240, w["concurrency"])
    print("  {:<34} --concurrency={:<3} -> {:>3} instances   ({})".format(name, w["concurrency"], n, w["note"]))
print()
print("Setting it wrong is the classic AI scaling bug:")
print("  I/O-bound app with --concurrency=1   -> {} instances for 240 waiting requests (huge waste)".format(instances_needed(240, 1)))
print("  compute-bound app with --concurrency=80 -> 80 requests fight over one GPU (all slow, or out of memory)")
print()
print("Set concurrency HIGH for an API-calling (I/O-bound) app, LOW for an on-instance model.")

# Output:
# Serving 240 concurrent requests - set --concurrency by workload:
#   calls an LLM API (I/O-bound)       --concurrency=80  ->   3 instances   (one instance holds many waiting requests)
#   runs a model on-instance (compute) --concurrency=4   ->  60 instances   (each request saturates the CPU/GPU)
#
# Setting it wrong is the classic AI scaling bug:
#   I/O-bound app with --concurrency=1   -> 240 instances for 240 waiting requests (huge waste)
#   compute-bound app with --concurrency=80 -> 80 requests fight over one GPU (all slow, or out of memory)
#
# Set concurrency HIGH for an API-calling (I/O-bound) app, LOW for an on-instance model.

- For 240 concurrent requests, an **I/O-bound** app at `--concurrency=80` needs just **3 instances** (each holds many waiting requests).
- A **compute-bound** on-instance model at `--concurrency=4` needs **60 instances** (each request saturates a GPU).
- Set it backwards and both break: concurrency 1 on the I/O app spins up **240 instances** (huge waste)…
- …and concurrency 80 on the model makes **eighty requests fight over one GPU** (all slow, or out of memory).

#### 💡 What just happened

⚡ What just happened? Cloud Run scales on request concurrency, and the right --concurrency depends on the workload: high (80-200) for an I/O-bound API-calling app, low (1-4) for a compute-bound on-instance model. Tradeoff: a single setting swings your fleet size by 20x and, set wrong, either wastes money or melts the GPU. Match concurrency to whether the app waits or computes.

- A concurrency slider and an I/O-bound vs compute-bound toggle
- The instances-needed readout for 240 concurrent flips between 3 and 60, with a GPU-fight warning

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Cold Starts, Min and Max Instances

### Cold Starts, Min and Max Instances

Scale-to-zero is cheap but the first request pays a cold start; min-instances keeps warm capacity so none do. Max-instances caps cost and, crucially, protects the downstream provider’s rate limit.

Two dials bound the fleet. **Scale-to-zero** (`--min-instances=0`) costs nothing when idle, but the first request after a quiet spell pays a **cold start** - and for an AI service that loads a model or heavy dependencies, that is not milliseconds but **10 to 40 seconds**. For anything latency-sensitive, set **`--min-instances`** to 1 or more so there is always a warm instance ready, and `--cpu-boost` to speed the starts that do happen. At the top end, **`--max-instances`** caps cost, but for an AI app it does something more important: it **protects the downstream provider**. You can only drive as many tokens per minute as the provider’s rate limit allows, so there is a number of instances beyond which the extra ones just collect 429s. Setting max-instances to that number keeps your fleet under the ceiling. The block models both, keyless.

> 🔥 **Analogy**
>
> It is **keeping one oven preheated, and knowing your kitchen’s limit**. If you turn every oven off overnight to save gas (scale to zero), the first breakfast order waits half an hour for an oven to heat (the cold start); keep one oven always on (min-instances) and that order goes straight in. And there is a hard cap on covers: however many chefs you hire, the single pass where plates leave the kitchen (the provider’s rate limit) only moves so many an hour - staff past that and they just crowd the pass (429s). Max-instances is staffing to the pass, not beyond it.

Your AI service scales to zero overnight. The first user in the morning waits 30 seconds for a response. What setting fixes it?

**📝 `04_cold_starts.py`** - *runnable*

In [ ]:
# Scale-to-zero saves money but the first request after idle pays a COLD START.
COLD_START_S = 30    # loading the model / heavy deps on a fresh instance
WARM_S = 0.2

print("Cold starts - the first request after the service scaled to zero:")
print("  --min-instances=0 (scale to zero): first request waits ~{:.0f}s (cold start), then fast; no idle cost".format(COLD_START_S))
print("  --min-instances=1 (keep 1 warm):   first request ~{:.1f}s (no cold start); a small idle cost".format(WARM_S))
print()

# --max-instances caps cost AND protects the downstream provider's rate limit.
PROVIDER_TPM = 200000                 # tokens/min the provider allows
TOKENS_PER_INSTANCE_PER_MIN = 20000   # tokens/min one instance drives
safe_max = PROVIDER_TPM // TOKENS_PER_INSTANCE_PER_MIN

print("--max-instances protects the downstream provider (and caps cost):")
print("  provider allows {:,} tokens/min; each instance drives ~{:,} tokens/min".format(PROVIDER_TPM, TOKENS_PER_INSTANCE_PER_MIN))
print("  so --max-instances={} keeps the whole fleet under the provider's rate limit".format(safe_max))
print("  scaling past that just returns 429s from the provider - more instances do NOT help.")

# Output:
# Cold starts - the first request after the service scaled to zero:
#   --min-instances=0 (scale to zero): first request waits ~30s (cold start), then fast; no idle cost
#   --min-instances=1 (keep 1 warm):   first request ~0.2s (no cold start); a small idle cost
#
# --max-instances protects the downstream provider (and caps cost):
#   provider allows 200,000 tokens/min; each instance drives ~20,000 tokens/min
#   so --max-instances=10 keeps the whole fleet under the provider's rate limit
#   scaling past that just returns 429s from the provider - more instances do NOT help.

- With **scale-to-zero**, the first request after idle waits about **30 seconds** for the model to load; then it is fast.
- With **min-instances=1**, a warm instance answers the first request in a fraction of a second - for a small idle cost.
- **Max-instances** caps cost and protects the provider: the fleet cannot drive more tokens/min than the provider allows.
- Here the provider’s limit works out to **10 instances** - scaling past that just returns 429s, so more instances do not help.

#### 💡 What just happened

⚡ What just happened? Scale-to-zero is cheap but the first request pays a 10-40s cold start; min-instances keeps warm capacity so none do, and max-instances caps cost while protecting the downstream provider’s rate limit. Tradeoff: a warm instance costs a little when idle, in exchange for no cold-start pain; a max cap trades peak throughput for not drowning in 429s. Next: what to do when a burst exceeds even a well-tuned fleet.

- A scale-to-zero timeline: the first request waits 30s cold, then fast; min=1 stays warm
- A max-instances slider bumps the provider rate-limit ceiling line

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Queue-Based Load Leveling

### Queue-Based Load Leveling

A burst above capacity should be enqueued, not dropped. Producers write to a queue, workers drain it at a steady rate, and you autoscale the workers on queue depth - so a spike is absorbed instead of lost.

Autoscaling takes time to react, and some bursts arrive faster than any fleet can grow. Rather than **drop** the overflow, **enqueue** it. In this pattern - classic **queue-based load leveling** - the front end just writes each job to a **queue** (Google Cloud Pub/Sub, AWS SQS) and returns immediately; a pool of **workers** pulls jobs and processes them at whatever *steady* rate the model tier can sustain. The queue **decouples the arrival rate from the processing rate**: a spike piles up in the queue and drains over the next few seconds or minutes, and *nothing is dropped*. And you autoscale the workers on the one signal that matters here - **queue depth**, the backlog - adding workers when the queue grows and removing them when it drains (this is what **KEDA** does on **Kubernetes**), with a `maxReplicaCount` to protect downstream. The block runs a burst through a queue, keyless.

> 🍿 **Analogy**
>
> It is the **ticket dispenser at a busy deli counter**. When a lunch rush pours in all at once, you do not turn people away because all the staff are busy - everyone takes a numbered ticket (the queue) and the counter serves them in order at a steady pace. The rush is absorbed into the ticket line and worked off over the next few minutes; nobody is lost, they just wait a little. And if the line gets long, you call in more staff (autoscale workers on the backlog) - up to however many can fit behind the counter (the max cap).

A burst of 500 hits a tier that can process 100 at once, and you put a queue in front. What happens to the other 400?

**📝 `05_queue_leveling.py`** - *runnable*

In [ ]:
# A burst above capacity: drop it, or QUEUE it and drain at a steady rate.
BURST = 500           # requests arriving in one spike
CAPACITY = 100        # requests the synchronous tier can handle at once

print("A burst of {} requests hits a tier that can handle {} at once:".format(BURST, CAPACITY))
overflow = max(0, BURST - CAPACITY)
print("  WITHOUT a queue: {} served now, {} OVERFLOW -> dropped / timed out".format(CAPACITY, overflow))
print()

# WITH a queue: enqueue everything; autoscale workers on QUEUE DEPTH (not CPU).
TARGET_PER_WORKER = 50    # aim for ~50 queued items per worker
MAX_WORKERS = 4          # maxReplicaCount - caps cost and protects downstream
def workers_for(depth):
    return min(MAX_WORKERS, -(-depth // TARGET_PER_WORKER))

print("  WITH a queue (autoscale workers on queue DEPTH, capped at {} workers):".format(MAX_WORKERS))
depth, tick = BURST, 0
while depth > 0 and tick < 8:
    w = workers_for(depth)
    drained = min(depth, w * TARGET_PER_WORKER)
    print("    tick {}: depth {:>3} -> {} workers -> drain {} -> depth {}".format(tick, depth, w, drained, depth - drained))
    depth -= drained
    tick += 1
print("  the queue ABSORBS the spike: nothing is dropped, the work is just delayed a little.")

# Output:
# A burst of 500 requests hits a tier that can handle 100 at once:
#   WITHOUT a queue: 100 served now, 400 OVERFLOW -> dropped / timed out
#
#   WITH a queue (autoscale workers on queue DEPTH, capped at 4 workers):
#     tick 0: depth 500 -> 4 workers -> drain 200 -> depth 300
#     tick 1: depth 300 -> 4 workers -> drain 200 -> depth 100
#     tick 2: depth 100 -> 2 workers -> drain 100 -> depth 0
#   the queue ABSORBS the spike: nothing is dropped, the work is just delayed a little.

- **Without a queue**, a 500-item burst on a 100-capacity tier **overflows by 400** - dropped or timed out.
- **With a queue**, all 500 are enqueued and workers autoscale on **queue depth** (capped at 4 workers).
- The backlog drains tick by tick - 500 → 300 → 100 → 0 - as the capped worker pool works it off.
- Nothing is dropped: the queue **absorbs the spike** and the work is just delayed a little.

#### 💡 What just happened

⚡ What just happened? Queue-based load leveling puts a queue between the burst and the workers: the front end enqueues and returns, workers drain at a steady rate, and you autoscale the workers on queue depth (KEDA) with a max cap. Tradeoff: requests wait a little longer under a spike, in exchange for never dropping work and never overwhelming the model tier. It decouples the arrival rate from the processing rate. Next: scaling the model tier itself.

- Requests pour in faster than the tier drains, overflowing; toggle a queue
- The backlog fills, workers autoscale on depth (capped), and the queue drains tick by tick

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Scaling the Model Tier: GPU Batching

### Scaling the Model Tier: GPU Batching

For a self-hosted model, throughput comes from batching requests together. vLLM’s continuous batching and PagedAttention lift GPU utilization from about 30 to 85 percent, and you autoscale GPU replicas on queue depth.

If you host the model yourself on a GPU, scaling is a different game - and the lever is **batching**. A naive serving loop handles one request at a time, which leaves the GPU idle between requests at around **30 percent** utilization. **vLLM** changes that with two ideas. **Continuous batching** schedules work at the token level, packing many requests into each GPU step so a short request is not stuck behind a long one, which lifts utilization to about **85 percent**. And **PagedAttention** stores the **KV cache** in fixed-size pages (like an operating system’s virtual memory), cutting the memory waste from 60 to 80 percent down to near zero, so you can fit **two to three times** more concurrent requests on the same GPU. Together they serve roughly **3 to 5 times** the throughput of the naive loop - on the identical hardware. Then you autoscale GPU **replicas** on queue depth and scale to zero when idle. The block compares naive and batched, keyless.

> 🚌 **Analogy**
>
> It is a **bus that waits to fill up versus one seat per trip**. The naive loop drives a whole bus across town for a single passenger, then comes back for the next - the engine (the GPU) runs the whole time but the bus is nearly empty. Continuous batching fills the seats: the bus picks up everyone heading the same way and makes the trip once, so the same engine moves far more people. And packing the seats tightly with no wasted rows (PagedAttention on the KV cache) fits even more aboard. Same bus, same fuel, several times the passengers.

Your self-hosted model sits at 30 percent GPU utilization serving one request at a time. What raises throughput most on the same GPU?

**📝 `06_gpu_batching.py`** - *runnable*

In [ ]:
# A self-hosted model on one GPU. Throughput comes from BATCHING requests together.
def gpu(mode):
    if mode == "naive":     # one request at a time
        return {"util": 30, "concurrent": 8,  "tok_per_s": 2600}
    return {"util": 85, "concurrent": 24, "tok_per_s": 9000}   # continuous batching + PagedAttention

naive, batched = gpu("naive"), gpu("batched")
print("One GPU, self-hosted model - naive loop vs continuous batching (vLLM):")
print("  {:<32} GPU {:>2}%   {:>2} concurrent   {:,} tokens/s".format("naive (one request at a time)", naive["util"], naive["concurrent"], naive["tok_per_s"]))
print("  {:<32} GPU {:>2}%   {:>2} concurrent   {:,} tokens/s".format("continuous batching (vLLM)", batched["util"], batched["concurrent"], batched["tok_per_s"]))
print()
print("  batching lifts GPU utilization {}% -> {}% and throughput ~{:.1f}x on the SAME GPU.".format(
    naive["util"], batched["util"], batched["tok_per_s"] / naive["tok_per_s"]))
print("  PagedAttention pages the KV cache, so memory waste drops from ~60% to near zero (2-3x more concurrent).")
print("  Then autoscale GPU REPLICAS on queue depth, and scale to zero when idle.")

# Output:
# One GPU, self-hosted model - naive loop vs continuous batching (vLLM):
#   naive (one request at a time)    GPU 30%    8 concurrent   2,600 tokens/s
#   continuous batching (vLLM)       GPU 85%   24 concurrent   9,000 tokens/s
#
#   batching lifts GPU utilization 30% -> 85% and throughput ~3.5x on the SAME GPU.
#   PagedAttention pages the KV cache, so memory waste drops from ~60% to near zero (2-3x more concurrent).
#   Then autoscale GPU REPLICAS on queue depth, and scale to zero when idle.

- The **naive loop** (one request at a time) leaves the GPU at about **30 percent** utilization.
- **Continuous batching** (vLLM) packs many requests per step and lifts utilization to about **85 percent**.
- That is roughly **3.5x the throughput** on the *same* GPU; **PagedAttention** pages the KV cache so memory waste drops to near zero.
- Then autoscale GPU **replicas on queue depth**, and scale to zero when the GPUs are idle.

#### 💡 What just happened

⚡ What just happened? For a self-hosted model, throughput comes from batching: vLLM’s continuous batching lifts GPU utilization from about 30 to 85 percent and PagedAttention pages the KV cache to fit 2-3x more concurrent requests, for roughly 3-5x the throughput on the same GPU. Tradeoff: batching adds a little scheduling latency, in exchange for several times the throughput. Then autoscale GPU replicas on queue depth. Deep serving internals are Module 9 / Module 14.

- A GPU cell grid fills to 30 percent (naive) vs 85 percent (continuous batching)
- A throughput bar grows about 3.5x; a KV-cache waste meter drops to near zero

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: The Scaling Ceiling: Backpressure

### The Scaling Ceiling: Backpressure

Autoscaling, queues, and batching raise the ceiling, but there is always a hard one - the provider’s rate limit or your GPU cap. Past it, adding instances does not help; you must shed load gracefully instead of collapsing.

Everything so far pushes the ceiling *up*, but it never removes it. There is always a **hard ceiling**: the provider’s rate limit, or the cap of your GPU fleet. Past that point, **adding instances does not help** - the extra ones just collect 429s from the provider. So the last piece of a scaling design is what to do *at* the ceiling: **backpressure**. Rather than let an overload drag *every* request into a timeout (a total collapse), you deliberately **shed the excess**: rate-limit at the door, return a **429 with a `Retry-After`** header, or **degrade** gracefully (fall back to a smaller model or a cached answer). The requests you accept stay fast; the ones you cannot serve fail *quickly and honestly* instead of hanging. It is the difference between a full restaurant seating who it can and turning the rest away politely, versus letting the whole floor grind to a halt. The block contrasts the two, keyless.

> 🚪 **Analogy**
>
> It is a **bouncer holding the line at a venue that is at capacity**. The fire code sets a hard limit on how many people can be inside (the rate-limit ceiling) - no amount of wanting more guests changes it. A good bouncer keeps exactly that many inside having a great time, and holds the rest in an orderly line with an honest “about twenty minutes” (a 429 with Retry-After). A bad night is no bouncer at all: everyone crams the doorway, nobody can move, and the people already inside cannot even get to the bar. Backpressure is the bouncer.

Demand climbs to 800 a second against a hard ceiling of 500. With no backpressure, what happens to the requests?

**📝 `07_backpressure.py`** - *runnable*

In [ ]:
# Autoscaling + queues + batching raise the ceiling, but there is ALWAYS a hard ceiling
# (the provider rate limit / your GPU fleet cap). Past it, more instances do not help.
CEILING = 500    # req/s the whole system can actually serve

print("Demand climbs toward the hard ceiling of {} req/s (provider rate limit / GPU cap):".format(CEILING))
for demand in [300, 500, 800]:
    if demand <= CEILING:
        print("  demand {:>3}: served {}, under the ceiling - fine".format(demand, demand))
        continue
    excess = demand - CEILING
    print("  demand {:>3}: over the ceiling by {}".format(demand, excess))
    print("      no backpressure  -> the overload times out ALL {} requests (collapse)".format(demand))
    print("      with backpressure -> serve {} fast, shed {} with 429 + Retry-After (or a cached/degraded reply)".format(CEILING, excess))
print()
print("You cannot scale past the ceiling. When you hit it, SHED load gracefully instead of collapsing")
print("(the retry / degradation mechanics are Lesson 12.2; the rate-limit config is Lesson 12.3).")

# Output:
# Demand climbs toward the hard ceiling of 500 req/s (provider rate limit / GPU cap):
#   demand 300: served 300, under the ceiling - fine
#   demand 500: served 500, under the ceiling - fine
#   demand 800: over the ceiling by 300
#       no backpressure  -> the overload times out ALL 800 requests (collapse)
#       with backpressure -> serve 500 fast, shed 300 with 429 + Retry-After (or a cached/degraded reply)
#
# You cannot scale past the ceiling. When you hit it, SHED load gracefully instead of collapsing
# (the retry / degradation mechanics are Lesson 12.2; the rate-limit config is Lesson 12.3).

- Below the **500-a-second ceiling**, everything is served normally.
- At **800**, **without backpressure** the overload times out *all* 800 of them - a collapse.
- **With backpressure**, the 500 the system can handle stay **fast**, and the 300 excess are **shed with a 429 + Retry-After** (or a cached/degraded reply).
- You cannot scale past the ceiling - so at it, shed load gracefully. The retry/degradation mechanics are Lesson 12.2.

**📝 `scaling-config (Cloud Run + KEDA + vLLM)`** - *illustrative*

In [ ]:
# SCALING CONFIG - an illustrative minimal example (Cloud Run + KEDA + vLLM).

# --- The stateless web/API tier: autoscale on CONCURRENCY, keep 1 warm, cap the fleet ---
#   gcloud run deploy ai-svc \
#     --concurrency=80 \            # I/O-bound (calls an LLM API): one instance holds many waiting requests
#     --min-instances=1 \          # keep 1 warm -> no cold start on the first request
#     --max-instances=10 \         # cap the fleet under the provider rate limit (200k TPM / 20k per instance)
#     --cpu-boost                  # faster cold start when it does scale up

# --- The queue-worker tier: autoscale on QUEUE DEPTH with KEDA (Kubernetes), scaledobject.yaml ---
#   apiVersion: keda.sh/v1alpha1
#   kind: ScaledObject
#   spec:
#     scaleTargetRef: {name: llm-worker}
#     minReplicaCount: 0           # scale to zero when the queue is empty
#     maxReplicaCount: 4           # protect downstream; cap cost
#     triggers:
#       - type: gcp-pubsub          # scale on the Pub/Sub backlog, not CPU
#         metadata: {subscriptionName: jobs-sub, mode: NumUndeliveredMessages, value: "50"}   # ~50 queued per worker

# --- The model tier: vLLM for continuous batching + PagedAttention on one GPU ---
#   vllm serve meta-llama/Llama-3.1-8B-Instruct \
#     --max-num-seqs 24 \          # continuous batching: up to 24 requests packed per step
#     --gpu-memory-utilization 0.9 # PagedAttention packs the KV cache tightly
# Output: (illustrative minimal example - needs gcloud, a KEDA-enabled cluster, and a GPU; not run locally.)

- The **web/API tier**: `--concurrency=80` (I/O-bound), `--min-instances=1` (no cold start), `--max-instances=10` (under the provider ceiling) - Steps 3 and 4.
- The **queue-worker tier**: a KEDA `ScaledObject` autoscales on `NumUndeliveredMessages` (queue depth), scale-to-zero, capped at 4 replicas - Step 5.
- The **model tier**: `vllm serve` with `--max-num-seqs` (continuous batching) and a high `--gpu-memory-utilization` (PagedAttention) - Step 6.
- Each tier scales on *its own* right signal - concurrency, queue depth, and batch fullness - which is the whole lesson in one file.

#### 💡 What just happened

⚡ What just happened? There is always a hard ceiling (the provider rate limit / your GPU cap); past it, adding instances just collects 429s, so you apply backpressure - rate-limit, return 429 + Retry-After, or degrade - shedding the excess so accepted requests stay fast instead of the whole system collapsing. Tradeoff / the whole lesson: you cannot serve infinite load, so you choose to fail a few fast and honestly rather than everyone slowly. The degradation mechanics are Lesson 12.2.

- Demand climbs past a ceiling line; toggle backpressure
- Without it every request times out (collapse); with it the ceiling is served fast and the excess is shed with 429

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> Scaling an AI app is a stack of pieces, each scaling on *its own* right signal. It starts by rejecting the normal playbook: an LLM app is **I/O-bound**, so CPU stays low and you scale on **concurrency**, with the provider’s rate limit as the real ceiling (Step 1). You scale the web tier **horizontally**, which demands **stateless** instances (Step 2), autoscale it on **concurrency set by workload** - high for I/O-bound, low for a compute-bound model (Step 3) - keep it **warm with min-instances** and **capped with max-instances** under the provider (Step 4). A **queue** levels bursts, its workers autoscaled on **queue depth** (Step 5); the **model tier** gets its throughput from **vLLM batching** and scales replicas on queue depth (Step 6); and at the unavoidable hard **ceiling**, **backpressure** sheds load gracefully rather than collapsing (Step 7). Ask five questions of any scaling design: is it on the **right signal**, **stateless**, **warm**, **capped to the provider**, **buffered by a queue**, and graceful at the ceiling?

|  | Web / API tier | Model tier (self-hosted) |
|---|---|---|
| Scale direction | horizontal (more instances) | vertical (bigger GPU) or replicas |
| Autoscale signal | request concurrency | queue depth / GPU utilization |
| Concurrency setting | high (I/O-bound, 80-200) | low (compute-bound, 1-4) |
| Throughput lever | more stateless instances | continuous batching (vLLM) |
| The ceiling | the provider’s rate limit | the GPU fleet cap |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now scale an AI app on the right signals. **Monitoring** those signals - RPS, concurrency, queue depth, p95 latency, GPU utilization - you meet in Lesson 12.8, and **cost** at scale (cost-per-token, batching economics, spot vs managed) comes in Module 13. The **backpressure** degradation mechanics (retry, circuit breaker, graceful fallback) are Lesson 12.2, and the request-aware routing and RPM/TPM limits that set the ceiling are Lesson 12.3. The primary references are [Cloud Run instance autoscaling](https://docs.cloud.google.com/run/docs/about-instance-autoscaling), the [AI cold-starts guide](https://cloud.google.com/blog/topics/developers-practitioners/a-guide-to-ai-cold-starts-on-cloud-run), [KEDA](https://keda.sh/) for queue-depth autoscaling, [vLLM](https://docs.vllm.ai/en/latest/) for continuous batching, and [Ray Serve](https://github.com/ray-project/ray) for autoscaled model replicas.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **ScalingAI Apps**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-12_7.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-12.7.html` - regenerate this notebook whenever the source HTML is updated.*
